# OCR KTP - Google Colab + Ngrok

Notebook ini menjalankan aplikasi Flask OCR KTP dan membuatnya dapat diakses via internet melalui Ngrok.

**Flow:** Upload Foto → Preprocessing (OpenCV) → OCR (PaddleOCR) → Tampilkan Hasil

In [ ]:
!git clone https://github.com/Herutriana44/OCR_KTP

## 1. Install Dependencies

In [ ]:
# Install semua dependency (proses ini memakan waktu ~2-5 menit)
!pip install -q flask werkzeug opencv-python-headless numpy pillow pyngrok
!pip install -q paddlepaddle==2.6.2 paddleocr==2.7.3
print("✅ Dependencies installed!")

## 2. Setup Ngrok (Gratis)

1. Daftar di [ngrok.com](https://ngrok.com) (gratis)
2. Copy **Auth Token** dari dashboard: https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste token di bawah:

In [ ]:
# Paste token ngrok Anda di sini (dari https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = ""  # Contoh: "2abc123xyz..."

if not NGROK_TOKEN:
    print("⚠️  Masukkan NGROK_TOKEN terlebih dahulu!")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok configured!")

## 3. Buat File Project

In [ ]:
import os
os.makedirs("/content/OCR_KTP/templates", exist_ok=True)
os.makedirs("/content/OCR_KTP/uploads", exist_ok=True)
os.makedirs("/content/OCR_KTP/processed", exist_ok=True)
print("✅ Directories created")

In [ ]:
%%writefile /content/OCR_KTP/preprocessing.py
"""Image Preprocessing untuk KTP - OpenCV"""
import cv2
import numpy as np
from typing import Optional

def preprocess_ktp_image(image_path: str) -> Optional[np.ndarray]:
    try:
        img = cv2.imread(image_path)
        if img is None:
            return None
        max_dim = 1200
        h, w = img.shape[:2]
        if max(h, w) > max_dim:
            scale = max_dim / max(h, w)
            img = cv2.resize(img, (int(w * scale), int(h * scale)))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.bilateralFilter(gray, 9, 75, 75)
        binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        kernel = np.ones((2, 2), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        max_area, best_contour = 0, None
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > img.shape[0] * img.shape[1] * 0.1:
                x, y, w, h = cv2.boundingRect(contour)
                aspect_ratio = w / float(h) if h > 0 else 0
                if 1.2 < aspect_ratio < 2.2 and area > max_area:
                    max_area, best_contour = area, contour
        if best_contour is not None:
            x, y, w, h = cv2.boundingRect(best_contour)
            padding = 10
            cropped = img[max(0,y-padding):min(img.shape[0],y+h+padding), max(0,x-padding):min(img.shape[1],x+w+padding)]
            return cropped
        return cv2.convertScaleAbs(img, alpha=1.2, beta=10)
    except Exception as e:
        print(f"Error: {e}")
        return None

def enhance_for_ocr(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image.copy()
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(enhanced, None, 10, 7, 21)
    return cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)

In [ ]:
%%writefile /content/OCR_KTP/ocr_processor.py
"""OCR Processor - PaddleOCR"""
import re
from typing import Dict, List, Tuple
import numpy as np

def extract_ktp_info(ocr_results: List) -> Dict[str, str]:
    all_texts = []
    for line in ocr_results:
        if line and len(line) >= 2:
            box, text_info = line[0], line[1]
            text = text_info[0] if isinstance(text_info, (list, tuple)) else str(text_info)
            y_center = np.mean([p[1] for p in box])
            all_texts.append((y_center, text.strip()))
    all_texts.sort(key=lambda x: x[0])
    full_text = " ".join([t[1] for t in all_texts])
    lines = [t[1] for t in all_texts]
    result = {"NIK":"","Nama":"","Tempat Lahir":"","Tanggal Lahir":"","Jenis Kelamin":"","Alamat":"","RT/RW":"","Kel/Desa":"","Kecamatan":"","Agama":"","Status Perkawinan":"","Pekerjaan":"","Kewarganegaraan":"","Berlaku Hingga":"","raw_text":full_text,"raw_lines":lines}
    nik_match = re.search(r'\b(\d{16})\b', full_text)
    if nik_match: result["NIK"] = nik_match.group(1)
    for line in lines:
        line_upper = line.upper()
        if "NAMA" in line_upper and len(line) > 5:
            parts = line.split(maxsplit=1)
            result["Nama"] = parts[1].strip() if len(parts) >= 2 and parts[0].upper() == "NAMA" else line.replace("NAMA", "").replace("Nama", "").strip()
        if "NIK" in line_upper:
            m = re.search(r'\d{16}', line)
            if m: result["NIK"] = m.group()
        if "LAHIR" in line_upper or "TEMPAT" in line_upper:
            clean = re.sub(r'^(TEMPAT|TGL|LAHIR|Tempat|Tgl)\s*[:.]?\s*', '', line, flags=re.I).strip()
            if re.search(r'\d{2}[-/]\d{2}[-/]\d{4}', clean):
                dm = re.search(r'(\d{2}[-/]\d{2}[-/]\d{4})', clean)
                if dm:
                    result["Tanggal Lahir"] = dm.group(1)
                    result["Tempat Lahir"] = re.sub(r'\d{2}[-/]\d{2}[-/]\d{4}', '', clean).strip(' ,-')
            elif clean and not result["Tempat Lahir"]: result["Tempat Lahir"] = clean
        if "LAKI-LAKI" in line_upper or "LAKI LAKI" in line_upper: result["Jenis Kelamin"] = "LAKI-LAKI"
        elif "PEREMPUAN" in line_upper: result["Jenis Kelamin"] = "PEREMPUAN"
        if "ALAMAT" in line_upper and len(line) > 7: result["Alamat"] = line.replace("ALAMAT", "").replace("Alamat", "").strip(' :')
        if "RT" in line_upper and "RW" in line_upper:
            rr = re.search(r'(\d{3})[/\s]*(\d{3})', line)
            if rr: result["RT/RW"] = f"{rr.group(1)}/{rr.group(2)}"
        if "KEL" in line_upper and "DESA" in line_upper: result["Kel/Desa"] = re.sub(r'^(KEL/DESA|Kel/Desa)\s*[:.]?\s*', '', line, flags=re.I).strip()
        if "KECAMATAN" in line_upper and len(line) > 10: result["Kecamatan"] = line.replace("KECAMATAN", "").replace("Kecamatan", "").strip(' :')
        if "AGAMA" in line_upper and len(line) > 6: result["Agama"] = line.replace("AGAMA", "").replace("Agama", "").strip(' :')
        if "KAWIN" in line_upper or "PERKAWINAN" in line_upper:
            if "BELUM" in line_upper: result["Status Perkawinan"] = "Belum Kawin"
            elif "KAWIN" in line_upper: result["Status Perkawinan"] = "Kawin"
            elif "CERAI" in line_upper: result["Status Perkawinan"] = "Cerai"
        if "PEKERJAAN" in line_upper and len(line) > 10: result["Pekerjaan"] = line.replace("PEKERJAAN", "").replace("Pekerjaan", "").strip(' :')
        if "KEWARGANEGARAAN" in line_upper or "WNI" in line_upper: result["Kewarganegaraan"] = "WNI" if "WNI" in line_upper else line.replace("KEWARGANEGARAAN", "").strip(' :')
        if "BERLAKU" in line_upper or "SEUMUR" in line_upper:
            if "SEUMUR" in line_upper or "HIDUP" in line_upper: result["Berlaku Hingga"] = "SEUMUR HIDUP"
            else:
                dm = re.search(r'(\d{2}[-/]\d{2}[-/]\d{4})', line)
                if dm: result["Berlaku Hingga"] = dm.group(1)
    return result

def run_ocr(image_path: str, use_angle_cls: bool = True) -> Tuple[List, Dict[str, str]]:
    try:
        from paddleocr import PaddleOCR
        ocr = PaddleOCR(use_angle_cls=use_angle_cls, lang='en', use_gpu=False, show_log=False)
        result = ocr.ocr(image_path, cls=use_angle_cls)
        if result is None or (isinstance(result, list) and len(result) == 0): return [], {}
        ocr_lines = []
        for page in result:
            if page:
                for item in page:
                    if item and len(item) >= 2: ocr_lines.append(item)
        return ocr_lines, extract_ktp_info(ocr_lines)
    except Exception as e:
        print(f"OCR Error: {e}")
        return [], {}

In [ ]:
%%writefile /content/OCR_KTP/app.py
"""Flask OCR KTP"""
import os, uuid
from flask import Flask, render_template, request, jsonify, send_from_directory
from werkzeug.utils import secure_filename
import cv2
from preprocessing import preprocess_ktp_image, enhance_for_ocr
from ocr_processor import run_ocr

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024
app.config['UPLOAD_FOLDER'] = '/content/OCR_KTP/uploads'
app.config['PROCESSED_FOLDER'] = '/content/OCR_KTP/processed'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
os.makedirs(app.config['PROCESSED_FOLDER'], exist_ok=True)
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'webp'}

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_file():
    file = request.files.get('file') or request.files.get('image')
    if not file or file.filename == '':
        return jsonify({'error': 'Tidak ada file'}), 400
    if not allowed_file(file.filename):
        return jsonify({'error': 'Format tidak didukung'}), 400
    try:
        filename = secure_filename(file.filename)
        unique_id = str(uuid.uuid4())[:8]
        save_filename = f"{unique_id}_{filename}"
        upload_path = os.path.join(app.config['UPLOAD_FOLDER'], save_filename)
        file.save(upload_path)
        processed_img = preprocess_ktp_image(upload_path)
        if processed_img is None: processed_img = cv2.imread(upload_path)
        if processed_img is None: return jsonify({'error': 'Gagal memproses gambar'}), 500
        enhanced_img = enhance_for_ocr(processed_img)
        processed_filename = f"proc_{unique_id}.jpg"
        processed_path = os.path.join(app.config['PROCESSED_FOLDER'], processed_filename)
        cv2.imwrite(processed_path, enhanced_img)
        ocr_results, ktp_info = run_ocr(processed_path, use_angle_cls=True)
        if not ktp_info.get('raw_text') and not ktp_info.get('NIK'):
            ocr_results, ktp_info = run_ocr(upload_path, use_angle_cls=True)
        display_info = {k: v for k, v in ktp_info.items() if k not in ('raw_text', 'raw_lines') and v}
        return jsonify({'success': True, 'ktp_data': display_info, 'raw_lines': ktp_info.get('raw_lines', []), 'processed_image': f'/processed/{processed_filename}', 'original_image': f'/uploads/{save_filename}'})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/uploads/<filename>')
def serve_upload(filename):
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

@app.route('/processed/<filename>')
def serve_processed(filename):
    return send_from_directory(app.config['PROCESSED_FOLDER'], filename)

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)

In [ ]:
%%writefile /content/OCR_KTP/templates/base.html
<!DOCTYPE html>
<html lang="id"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>OCR KTP</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>:root{--bg-primary:#0f0f12;--bg-secondary:#18181c;--bg-card:#1e1e24;--accent:#6366f1;--accent-hover:#818cf8;--accent-muted:rgba(99,102,241,.15);--text-primary:#f4f4f5;--text-secondary:#a1a1aa;--border:#27272a;--error:#ef4444}*{margin:0;padding:0;box-sizing:border-box}body{font-family:'DM Sans',sans-serif;background:var(--bg-primary);color:var(--text-primary);min-height:100vh;line-height:1.6}.container{max-width:900px;margin:0 auto;padding:2rem}header{text-align:center;margin-bottom:3rem}header h1{font-size:2rem;font-weight:700;background:linear-gradient(135deg,var(--text-primary),var(--accent));-webkit-background-clip:text;-webkit-text-fill-color:transparent}header p{color:var(--text-secondary);margin-top:.5rem}</style>
{% block extra_css %}{% endblock %}</head><body><div class="container"><header><h1>OCR KTP Indonesia</h1><p>Upload foto KTP untuk ekstraksi informasi</p></header>{% block content %}{% endblock %}</div>{% block extra_js %}{% endblock %}</body></html>

In [ ]:
%%writefile /content/OCR_KTP/templates/index.html
{% extends "base.html" %}
{% block extra_css %}
<style>.upload-zone{border:2px dashed var(--border);border-radius:16px;padding:3rem 2rem;text-align:center;background:var(--bg-secondary);cursor:pointer;margin-bottom:2rem}.upload-zone:hover,.upload-zone.dragover{border-color:var(--accent);background:var(--accent-muted)}.upload-zone input{display:none}.upload-icon{font-size:3rem;margin-bottom:1rem;opacity:.7}.upload-text{color:var(--text-secondary)}.upload-text strong{color:var(--accent)}.btn{display:inline-flex;align-items:center;padding:.75rem 1.5rem;border-radius:10px;font-weight:600;border:none;cursor:pointer;font-family:inherit}.btn-primary{background:var(--accent);color:#fff}.btn-primary:hover{background:var(--accent-hover)}.btn-primary:disabled{opacity:.6;cursor:not-allowed}.loading{display:none;text-align:center;padding:2rem}.loading.active{display:block}.spinner{width:48px;height:48px;border:4px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:spin .8s linear infinite;margin:0 auto 1rem}@keyframes spin{to{transform:rotate(360deg)}}.results{display:none;margin-top:2rem}.results.active{display:block}.results-grid{display:grid;grid-template-columns:1fr 1fr;gap:2rem;margin-top:1.5rem}@media(max-width:768px){.results-grid{grid-template-columns:1fr}}.image-preview{background:var(--bg-card);border-radius:12px;overflow:hidden;border:1px solid var(--border)}.image-preview img{width:100%;display:block}.image-preview figcaption{padding:.75rem 1rem;font-size:.85rem;color:var(--text-secondary);border-top:1px solid var(--border)}.ktp-data{background:var(--bg-card);border-radius:12px;padding:1.5rem;border:1px solid var(--border)}.ktp-data h3{font-size:1.1rem;margin-bottom:1rem;color:var(--accent)}.data-row{display:flex;padding:.6rem 0;border-bottom:1px solid var(--border);gap:1rem}.data-row:last-child{border-bottom:none}.data-label{font-weight:500;color:var(--text-secondary);min-width:140px}.data-value{color:var(--text-primary)}.raw-text{margin-top:2rem;background:var(--bg-card);border-radius:12px;padding:1rem;border:1px solid var(--border)}.raw-text h4{font-size:.9rem;color:var(--text-secondary);margin-bottom:.75rem}.raw-text pre{font-size:.8rem;color:var(--text-secondary);white-space:pre-wrap;word-break:break-all}.error-msg{background:rgba(239,68,68,.1);border:1px solid var(--error);color:#fca5a5;padding:1rem;border-radius:10px;margin-top:1rem;display:none}.error-msg.active{display:block}</style>
{% endblock %}
{% block content %}
<form id="uploadForm"><div class="upload-zone" id="uploadZone"><input type="file" id="fileInput" name="file" accept="image/png,image/jpeg,image/jpg,image/webp"><div class="upload-icon">📄</div><p class="upload-text"><strong>Klik atau drag & drop</strong> foto KTP<br>Format: PNG, JPG (max 16MB)</p></div><div style="text-align:center"><button type="submit" class="btn btn-primary" id="submitBtn" disabled>Proses OCR</button></div></form>
<div class="loading" id="loading"><div class="spinner"></div><p>Memproses... (Preprocessing → OCR)</p></div>
<div class="error-msg" id="errorMsg"></div>
<div class="results" id="results"><h2 style="margin-bottom:1rem;font-size:1.25rem">Hasil Deteksi</h2><div class="results-grid"><div><figure class="image-preview"><img id="processedImg" src="" alt="Gambar diproses"><figcaption>Gambar setelah preprocessing</figcaption></figure></div><div class="ktp-data"><h3>Informasi KTP</h3><div id="ktpDataList"></div></div></div><div class="raw-text"><h4>Teks Terdeteksi (Raw OCR)</h4><pre id="rawText"></pre></div></div>
{% endblock %}
{% block extra_js %}
<script>
const uz=document.getElementById('uploadZone'),fi=document.getElementById('fileInput'),uf=document.getElementById('uploadForm'),sb=document.getElementById('submitBtn'),ld=document.getElementById('loading'),rs=document.getElementById('results'),em=document.getElementById('errorMsg'),kd=document.getElementById('ktpDataList'),rt=document.getElementById('rawText'),pi=document.getElementById('processedImg');
uz.onclick=()=>fi.click();uz.ondragover=e=>{e.preventDefault();uz.classList.add('dragover')};uz.ondragleave=()=>uz.classList.remove('dragover');uz.ondrop=e=>{e.preventDefault();uz.classList.remove('dragover');if(e.dataTransfer.files.length){fi.files=e.dataTransfer.files;sb.disabled=!fi.files.length}};
fi.onchange=()=>sb.disabled=!fi.files.length;
uf.onsubmit=async e=>{e.preventDefault();if(!fi.files.length)return;em.classList.remove('active');rs.classList.remove('active');ld.classList.add('active');sb.disabled=true;const fd=new FormData();fd.append('file',fi.files[0]);try{const r=await fetch('/upload',{method:'POST',body:fd});const d=await r.json();if(!r.ok)throw new Error(d.error||'Error');if(d.success){pi.src=d.processed_image;const L={'NIK':'NIK','Nama':'Nama','Tempat Lahir':'Tempat Lahir','Tanggal Lahir':'Tanggal Lahir','Jenis Kelamin':'Jenis Kelamin','Alamat':'Alamat','RT/RW':'RT/RW','Kel/Desa':'Kel/Desa','Kecamatan':'Kecamatan','Agama':'Agama','Status Perkawinan':'Status Perkawinan','Pekerjaan':'Pekerjaan','Kewarganegaraan':'Kewarganegaraan','Berlaku Hingga':'Berlaku Hingga'};let h='';for(const[k,v] of Object.entries(d.ktp_data))h+=`<div class="data-row"><span class="data-label">${L[k]||k}</span><span class="data-value">${(''+v).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;')}</span></div>`;kd.innerHTML=h||'<p style="color:var(--text-secondary)">Tidak ada data terdeteksi</p>';rt.textContent=d.raw_lines?.join('\n')||'';rs.classList.add('active');rs.scrollIntoView({behavior:'smooth'})}else throw new Error(d.error)}catch(err){em.textContent=err.message;em.classList.add('active')}finally{ld.classList.remove('active');sb.disabled=false}}
</script>
{% endblock %}

## 4. Jalankan Flask + Ngrok

In [ ]:
import threading
from pyngrok import ngrok

# Jalankan Flask di background thread
def run_flask():
    import sys
    sys.path.insert(0, '/content/OCR_KTP')
    os.chdir('/content/OCR_KTP')
    from app import app
    app.run(debug=False, host='0.0.0.0', port=5000, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Tunggu Flask siap
import time
time.sleep(5)

# Buka tunnel Ngrok ke port 5000
tunnel = ngrok.connect(5000)

print("="*60)
print("✅ OCR KTP berjalan!")
print("="*60)
print(f"\n🌐 Buka link ini di browser:")
print(f"\n   {tunnel.public_url}\n")
print("="*60)

In [ ]:
# Tampilkan URL (jika cell di atas sudah dijalankan)
try:
    from pyngrok import ngrok
    tunnels = ngrok.get_tunnels()
    if tunnels:
        print("🌐 URL publik:", tunnels[0].public_url)
    else:
        print("Jalankan cell sebelumnya terlebih dahulu.")
except:
    print("Jalankan cell 'Jalankan Flask + Ngrok' terlebih dahulu.")

## Catatan

- **Ngrok free**: URL akan berubah setiap kali Colab di-restart
- **Colab timeout**: Session akan timeout jika idle ~90 menit
- **PaddleOCR**: Pertama kali load model akan download (~100MB), proses OCR pertama mungkin lambat